
---
Methods Project Part 3
---


## Clustering using Raw Data vs. Learned Representations


In [31]:
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split

import numpy as np
import plotly.graph_objects as go

from keras.datasets import mnist
from keras import layers, models
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping

from typing import Literal

RANDOM_STATE = 1 

(X_train_mnist, y_train_mnist), (X_test_mnist, y_test_mnist) = mnist.load_data()

# We combine the train and test set to a single before masking
X_mnist = np.concatenate([X_train_mnist, X_test_mnist], axis=0)
y_mnist = np.concatenate([y_train_mnist, y_test_mnist], axis=0)

# We reshape the data to be in the format (num_samples, num_features) and normalize pixel values to [0, 1]
image_size = X_train_mnist.shape[1] #28 for MNIST
# (num_samples, 28, 28) -> (num_samples, 28, 28, 1) and normalize pixel values to [0, 1]
X_mnist = X_mnist.reshape(-1, image_size, image_size, 1).astype('float32') / 255

# Dataset A (digits 0-4), Dataset B (digits 5-9)
X_mnist_A,y_mnist_A = X_mnist[y_mnist <= 4], y_mnist[y_mnist <= 4]

# One-hot encoding for the labels
y_mnist_A = to_categorical(y_mnist_A, num_classes=5)

def get_2_layer_cnn_base():
    base = models.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten()
    ], name="base_2_layer_cnn")
    return base

def build_and_compile_full_model(base_model):
    """Attaches 2 Fully-Connected layers"""
    inputs = layers.Input(shape=(28, 28, 1))
    # Pass the inputs through the base feature extractor
    x = base_model(inputs)
    
    # Classification head with 2 fully connected layers
    x = layers.Dense(64, activation='relu', name="fc_1")(x)
    outputs = layers.Dense(5, activation='softmax', name="fc_2_out")(x)
    
    # Combine into a single model
    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

early_stopping = EarlyStopping(
    monitor='val_loss',         
    patience=3,                 
    restore_best_weights=True,  # Revert to the best model, not the last epoch's model
    verbose=1              
)

In [ ]:
# Split Dataset A into 80/20 set for training the CNN and for clustering evaluation respectively 
X_mnist_A_80, X_mnist_A_20, y_mnist_A_80, y_mnist_A_20 = train_test_split(X_mnist_A, y_mnist_A, test_size=0.2, random_state=RANDOM_STATE)
# Train 2-layer CNN on 80% of dataset A and 10% of it for validation (early stopping) 
print("Training 2-layer CNN Dataset A")
base_2 = get_2_layer_cnn_base()
model_2_data_A = build_and_compile_full_model(base_2)
model_2_data_A.fit(X_mnist_A_80, y_mnist_A_80, epochs=100, batch_size=128, verbose="auto", validation_split=0.1,callbacks=[early_stopping])

Training 2-layer CNN Dataset A
Epoch 1/100
202/202 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9511 - loss: 0.1625 - val_accuracy: 0.9874 - val_loss: 0.0404
Epoch 2/100
202/202 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9902 - loss: 0.0326 - val_accuracy: 0.9930 - val_loss: 0.0249
Epoch 3/100
202/202 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9934 - loss: 0.0212 - val_accuracy: 0.9934 - val_loss: 0.0183
Epoch 4/100
202/202 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9954 - loss: 0.0160 - val_accuracy: 0.9948 - val_loss: 0.0211
Epoch 5/100
202/202 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9956 - loss: 0.0131 - val_accuracy: 0.9962 - val_loss: 0.0138
Epoch 6/100
202/202 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9978 - loss: 0.0077 - val_accuracy: 0.9951 - val_loss: 0.0183
Epoch 7/100
202/202 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9970 - loss: 0.0088 - val_accuracy: 0.9955 - val_loss: 0.0144
Epoch 8/100
202/202 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0

While we use model_2_data_A to train the network (it needs the classification head to compute loss 
and update weights), we extract learned representations using only base_2 — the frozen convolutional 
base — which outputs the feature vector fed into the FC layers.

In [ ]:
# For raw representation we flatten the 28x28x1 images into 784-dimensional pixel vectors
X_raw = X_mnist_A_20.reshape(X_mnist_A_20.shape[0], -1)
# For learned representation we use the output of the base CNN (before the fully connected layers)
X_learned = base_2.predict(X_mnist_A_20)

print(f"Raw Data Shape: {X_raw.shape}")
print(f"Learned Representation Shape: {X_learned.shape}")

224/224 ━━━━━━━━━━━━━━━━━━━━ 0s 880us/step
Raw Data Shape: (7147, 784)
Learned Representation Shape: (7147, 1600)


In [ ]:
def evaluate_clustering(X_train,algo: Literal["kmeans", "spectral"],k):
    """
    Runs the specified clustering algorithm 10 times each with different random seeds and 
    returns the mean and standard deviation of the silhouette scores.
    """
    silhouette_scores = []
    for _ in range(10):
        # n_init=1 ensures randomization for each iteration
        if algo == 'kmeans':
            model = KMeans(n_clusters=k,n_init=1) 
        else:
            # nearest_neighbors affinity over default RBF for image datasets to prevent memory/time blowup
            # kmeans to perform final cluster assignment in the spectral embedding space
            model = SpectralClustering(n_clusters=k,n_init=1,n_jobs=-1,affinity='nearest_neighbors',assign_labels='kmeans')
        labels = model.fit_predict(X_train)
        silhouette_scores.append(silhouette_score(X_train, labels)) #Score ranges from[-1, 1]
    return np.mean(silhouette_scores), np.std(silhouette_scores)

In [ ]:
# k = 1 gives undefined for silhouette score, so we start from k=2
k_values = [2,3,4,5]

k_means_raw = {}
k_means_learned = {}
spectral_raw = {}
spectral_learned = {}

for k in k_values:
    print(f"  Evaluating K={k}...")
    
    # K-Means for both raw and learned representations
    k_means_raw[k] = evaluate_clustering(X_raw, "kmeans", k)
    k_means_learned[k] = evaluate_clustering(X_learned, "kmeans", k)
    
    # Spectral Clustering for both raw and learned representations
    spectral_raw[k] = evaluate_clustering(X_raw, "spectral", k)
    spectral_learned[k] = evaluate_clustering(X_learned, "spectral", k)
    
fig = go.Figure()
# K-Means Traces
fig.add_trace(go.Scatter(
    x=k_values, y=[k_means_raw[k][0] for k in k_values],
    error_y=dict(type='data', array=[k_means_raw[k][1] for k in k_values], visible=True),
    mode='lines+markers', name='K-Means (Raw)',
    marker=dict(symbol='circle', size=10, color='#636efa'),
    line=dict(dash='dash')
))

fig.add_trace(go.Scatter(
    x=k_values, y=[k_means_learned[k][0] for k in k_values],
    error_y=dict(type='data', array=[k_means_learned[k][1] for k in k_values], visible=True),
    mode='lines+markers', name='K-Means (Learned)',
    marker=dict(symbol='circle', size=10, color='#00cc96')
))

# Spectral Clustering Traces
fig.add_trace(go.Scatter(
    x=k_values, y=[spectral_raw[k][0] for k in k_values],
    error_y=dict(type='data', array=[spectral_raw[k][1] for k in k_values], visible=True),
    mode='lines+markers', name='Spectral (Raw)',
    marker=dict(symbol='diamond', size=10, color='#EF553B'),
    line=dict(dash='dash')
))

fig.add_trace(go.Scatter(
    x=k_values, y=[spectral_learned[k][0] for k in k_values],
    error_y=dict(type='data', array=[spectral_learned[k][1] for k in k_values], visible=True),
    mode='lines+markers', name='Spectral (Learned)',
    marker=dict(symbol='diamond', size=10, color='#ab63fa')
))

fig.update_layout(
    title='Clustering Performance: Raw Data vs. Learned Representations (Dataset A)',
    xaxis_title='Number of Clusters (K)',
    yaxis_title='Silhouette Coefficient (Mean +/- Std Dev)',
    xaxis=dict(
    tickvals=k_values,
    title='Number of Clusters (K)  [K=1 excluded: silhouette undefined]'),
    width=1000, 
    height=700,
    legend=dict(title='Algorithm & Data Source')
)


fig.show()

  Evaluating K=2...
  Evaluating K=3...
  Evaluating K=4...
  Evaluating K=5...
K-Means (Raw): {2: (np.float64(0.13155257999897002), np.float64(0.0035869381689964557)), 3: (np.float64(0.0976865291595459), np.float64(0.010337849613310187)), 4: (np.float64(0.1004680335521698), np.float64(0.0005247216067242352)), 5: (np.float64(0.10936369597911835), np.float64(0.013347262168075563))}
K-Means (Learned): {2: (np.float64(0.14180851131677627), np.float64(0.0005970974962320952)), 3: (np.float64(0.15880681276321412), np.float64(0.0015198121837037787)), 4: (np.float64(0.19428013414144515), np.float64(2.007186412811279e-06)), 5: (np.float64(0.19458871185779572), np.float64(0.017661937521679657))}
Spectral (Raw): {2: (np.float64(0.1370094269514084), np.float64(0.0)), 3: (np.float64(0.09326190501451492), np.float64(0.0)), 4: (np.float64(0.08853298425674438), np.float64(0.0)), 5: (np.float64(0.05928320437669754), np.float64(0.0020980548005016153))}
Spectral (Learned): {2: (np.float64(0.1531364321708